In [ ]:
## 用来观测整体的attention map和相应的report 
import argparse
import math
from pathlib import Path
from typing import Tuple

import lightning.pytorch as pl
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from tqdm import tqdm

# -----------------------------------------------------------------------------
# Project‑specific imports – make sure these are on PYTHONPATH
# -----------------------------------------------------------------------------
from dataset.data_module import DataModule  # noqa: E402
from models.R2GenGPT import R2GenGPT       # noqa: E402
from configs.config import parser as base_parser  # noqa: E402

# ================================ Defaults ===================================
DEFAULTS = dict(
    # --- data ----------------------------------------------------------------
    dataset="mimic_cxr",
    annotation="/data2/yuhaowang/MIMIC-CXR/mimic_annotation_all.json",
    base_dir="/data2/yuhaowang/MIMIC-CXR/files/",
    # --- checkpoint ----------------------------------------------------------
    delta_file="/home/yuhaowang/project/report_generation/TRRG/R2GenGPT/deep_checkpoint_step42310.pth",  # supply your own .ckpt or .pth
    # --- inference hyper‑params ---------------------------------------------
    batch_size=16,
    max_length=100,
    min_new_tokens=80,
    max_new_tokens=120,
    repetition_penalty=2.0,
    length_penalty=2.0,
    # --- device / output -----------------------------------------------------
    device="cuda:0",
    output_dir="/home/yuhaowang/project/report_generation/TRRG/R2GenGPT/outputs/mimic_cxr/attn_v1",
    # --- precision -----------------------------------------------------------
    fp16=False,  # if True → runs model & data in float16
)

# ========================== Helper functions =================================

def factorize_grid(n_patch: int) -> Tuple[int, int]:
    """Return (gh, gw) so that gh * gw == n_patch and |gh‑gw| minimal."""
    root = int(math.sqrt(n_patch))
    for h in range(root, 0, -1):
        if n_patch % h == 0:
            return h, n_patch // h
    return 1, n_patch  # fallback (unlikely)


def cls_attention_to_heatmap(cls_attn: torch.Tensor,
                             hw: Tuple[int, int],
                             grid_hw: Tuple[int, int] | None = None) -> torch.Tensor:
    """Convert CLS‑to‑patch attention to a [0,1] heat‑map of size ``hw``.

    * ``cls_attn`` – (N,) tensor, where N = #patches.
    * ``hw``        – (H, W) size of original image.
    * ``grid_hw``   – optional (gh, gw) patch grid size; if ``None`` it is
                      inferred automatically.
    """
    if grid_hw is None:
        grid_hw = factorize_grid(cls_attn.numel())
    gh, gw = grid_hw
    grid = cls_attn.view(1, 1, gh, gw)
    heat = F.interpolate(grid, size=hw, mode="bilinear", align_corners=False)
    heat = heat.squeeze().clamp_(0, 1)  # (H, W)
    heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-5)
    return heat


def save_overlay(img_tensor: torch.Tensor,
                 heatmap: torch.Tensor,
                 save_path: Path,
                 alpha: float = 0.45) -> None:
    """Save PIL image with heat‑map overlay."""
    img = TF.to_pil_image(img_tensor.cpu().clamp(0, 1))
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.imshow(heatmap.cpu(), cmap="inferno", alpha=alpha)
    plt.axis("off")
    plt.tight_layout(pad=0)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=300)
    plt.close()

# ======================= Monkey‑patch generate_reports =======================

def _add_generate_reports():
    """Attach `generate_reports` to **R2GenGPT** if it's missing."""
    if hasattr(R2GenGPT, "generate_reports"):
        return

    def _generate_reports(self: R2GenGPT, images, **gen_kwargs):
        # Encode images → embeddings
        img_embeds, atts_img = self.encode_img(images)
        img_embeds = self.layer_norm(img_embeds)
        img_embeds, atts_img = self.prompt_wrap(img_embeds, atts_img)

        # match dtype with LLaMA
        dtype = self.llama_model.dtype
        img_embeds = img_embeds.to(dtype)

        batch_size = img_embeds.shape[0]
        bos = (torch.ones([batch_size, 1], dtype=torch.long, device=img_embeds.device)
               * self.llama_tokenizer.bos_token_id)
        bos_embeds = self.embed_tokens(bos).to(dtype)
        atts_bos = atts_img[:, :1]

        inputs_embeds = torch.cat([bos_embeds, img_embeds], dim=1)
        attention_mask = torch.cat([atts_bos, atts_img], dim=1)

        outputs = self.llama_model.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            **gen_kwargs,
        )
        return [self.decode(o) for o in outputs]

    R2GenGPT.generate_reports = _generate_reports  # type: ignore

# ================================ Inference ==================================

def run_inference(args):
    pl.seed_everything(42)

    # 1️⃣ Data‑module
    dm = DataModule(args)
    dm.setup("test")
    test_loader = dm.test_dataloader()

    # 2️⃣ Model
    precision = torch.float16 if args.fp16 else torch.float32
    model = R2GenGPT(args).to(dtype=precision, device=args.device).eval()

    # 3️⃣ Output folders
    out_root = Path(args.output_dir)
    attn_dir = out_root / "attn_maps" / "test"
    rpt_dir = out_root / "reports" / "test"
    attn_dir.mkdir(parents=True, exist_ok=True)
    rpt_dir.mkdir(parents=True, exist_ok=True)

    # 4️⃣ Generation hyper‑parameters
    gen_kwargs = dict(
        max_new_tokens=args.max_new_tokens,
        min_new_tokens=args.min_new_tokens,
        repetition_penalty=args.repetition_penalty,
        length_penalty=args.length_penalty,
        num_beams=3,
        do_sample=False,  # greedy / beam search – no top_p/temperature
    )

    with torch.no_grad():
        for batch in tqdm(test_loader, total=len(test_loader), ncols=100):

            images = [img.to(args.device, dtype=precision) for img in batch["image"]]
            first_view = images[0]                        # Tensor, shape (B, C, H, W)
            study_ids  = batch["id"]                     # list[str]

            # 1️⃣ 生成报告文本 —— 原逻辑不变
            reports = model.generate_reports(images=images, **gen_kwargs)

            # 2️⃣ 计算全局注意力 (no-CLS) -------------------------------------- 🔧②
            # 让 visual_encoder 返回 hidden states
            ve_out = model.visual_encoder(
                first_view,
                output_hidden_states=True,   # 👈 关键
                return_dict=True
            )
            patch_embeds = ve_out.last_hidden_state       # (B, N, D)
            q_global = patch_embeds.mean(dim=1, keepdim=True)        # (B, 1, D)
            attn_logits = torch.matmul(                   # (B, 1, N)
                q_global, patch_embeds.transpose(-1, -2)
            ) / math.sqrt(patch_embeds.size(-1))
            global2patch = torch.softmax(attn_logits, dim=-1).squeeze(1)  # (B, N)

            # ------------------------------------------------------------------
            # (3) 保存热图 & 报告 —— 用新向量替换 cls2patch                 🔧③
            # ------------------------------------------------------------------
            _, _, H, W = first_view.shape
            N = global2patch.shape[1]
            gh, gw = factorize_grid(N)

            for img_tensor, heat_vec, fname, rpt in zip(first_view,
                                                        global2patch,
                                                        study_ids,
                                                        reports):
                heat = cls_attention_to_heatmap(heat_vec.float(),
                                                (H, W),
                                                (gh, gw))
                save_overlay(img_tensor.float(), heat,
                             attn_dir / f"{fname}.png")
                (rpt_dir / f"{fname}.txt").write_text(rpt + "\n",
                                                      encoding="utf-8")
     
# =============================== CLI parsing =================================

def build_parser() -> argparse.ArgumentParser:
    infer_parser = argparse.ArgumentParser(parents=[base_parser], add_help=False)

    # Merge defaults & add `--fp16` flag
    existing_opts = {opt for act in infer_parser._actions for opt in act.option_strings}
    for k, v in DEFAULTS.items():
        flag = f"--{k}"
        if flag in existing_opts:
            infer_parser.set_defaults(**{k: v})
        else:
            if isinstance(v, bool):
                infer_parser.add_argument(flag, action="store_true" if not v else "store_false")
            else:
                infer_parser.add_argument(flag, default=v, type=type(v))
    return infer_parser

# =================================== Main ===================================
if __name__ == "__main__":
    _add_generate_reports()
    parser = build_parser()
    cli_args = parser.parse_args()
    run_inference(cli_args)


## 推理Patch token和文本Token 的注意力关注程度

In [2]:
import json
annotation_path = '/data2/yuhaowang/MIMIC-CXR/mimic_annotation_all.json'
with open(annotation_path, 'r') as f:
    annotations = json.load(f)
annotations['test']

[{'id': '427446c1-881f5cce-85191ce1-91a58ba9-0a57d3f5',
  'study_id': 50051329,
  'subject_id': 10046166,
  'report': 'impression: No evidence of acute cardiopulmonary process. Findings: Lateral view somewhat limited due to overlying motion artifact. The\n lungs are low in volume.  There is no focal airspace consolidation to suggest\n pneumonia.  A 1.2-cm calcified granuloma just below the medial aspect of the\n right hemidiaphragm is unchanged from prior study.  No pleural effusions or\n pulmonary edema. There is no pneumothorax.\n \n The inferior sternotomy wire is fractured but unchanged. Surgical clips and\n vascular markers in the thorax are related to prior CABG surgery.',
  'image_path': ['p10/p10046166/s50051329/427446c1-881f5cce-85191ce1-91a58ba9-0a57d3f5.jpg'],
  'split': 'test'},
 {'id': 'abea5eb9-b7c32823-3a14c5ca-77868030-69c83139',
  'study_id': 50051329,
  'subject_id': 10046166,
  'report': 'impression: No evidence of acute cardiopulmonary process. Findings: Lateral vie